In [1]:
import pandas as pd

In [2]:
urban_flood_df = pd.read_csv("../../data/urban_flood_groundsource_df.csv")
urban_flood_df = urban_flood_df.drop(columns=["geo_level"]) # remove unnecessary column
urban_flood_df = urban_flood_df.rename(columns={"adm3_en": "city_name", "adm3_psgc": "psgc"})
urban_flood_df.head()

,city_name,psgc,start_date,end_date
0,City of Silay,604526000,2022-01-01,2022-01-02
1,City of Surigao,1606724000,2022-01-01,2022-01-02
2,City of Manila,1380600000,2022-01-09,2022-01-09
3,City of Tagum,1102319000,2022-01-18,2022-01-23
4,City of Tagum,1102319000,2022-01-18,2022-01-18


In [3]:
# Ensure columns are datetime objects
urban_flood_df["start_date"] = pd.to_datetime(urban_flood_df["start_date"])
urban_flood_df["end_date"] = pd.to_datetime(urban_flood_df["end_date"])

# Create a list of all dates in the range for each row
urban_flood_df["date_list"] = urban_flood_df.apply(
    lambda x: pd.date_range(start=x["start_date"], end=x["end_date"], freq="D"), 
    axis=1
)

# Explode the list into individual rows based on the date range
urban_flood_df_exploded = urban_flood_df.explode("date_list")
urban_flood_df_exploded = urban_flood_df_exploded.rename(columns={"date_list": "active_flood_day"})

urban_flood_df_exploded.head()

,city_name,psgc,start_date,end_date,active_flood_day
0,City of Silay,604526000,2022-01-01,2022-01-02,2022-01-01
0,City of Silay,604526000,2022-01-01,2022-01-02,2022-01-02
1,City of Surigao,1606724000,2022-01-01,2022-01-02,2022-01-01
1,City of Surigao,1606724000,2022-01-01,2022-01-02,2022-01-02
2,City of Manila,1380600000,2022-01-09,2022-01-09,2022-01-09


In [4]:
# Create Year-Month column for grouping
urban_flood_df_exploded["month_year"] = urban_flood_df_exploded["active_flood_day"].dt.to_period("M")

# Count UNIQUE days per city per month
urban_afd_df = urban_flood_df_exploded.groupby(
    ["city_name", "psgc", "month_year"]
)["active_flood_day"].nunique().reset_index()
urban_afd_df.rename(columns={"active_flood_day": "reported_active_flood_days"}, inplace=True)
urban_afd_df.head()

,city_name,psgc,month_year,reported_active_flood_days
0,Batangas City,401005000,2022-10,5
1,Batangas City,401005000,2022-11,1
2,Batangas City,401005000,2024-01,2
3,Batangas City,401005000,2024-07,3
4,Batangas City,401005000,2024-10,5


In [5]:
# Checking the target dataset counts
urban_afd_df.shape[0]

1408

In [6]:
# Checking the counts of cities, should align with the first notebook
urban_afd_df[["city_name", "psgc"]].nunique()

city_name    143
psgc         147
dtype: int64

In [7]:
# Save to CSV
urban_afd_df.to_csv("../../data/urban_active_flood_days.csv", index=False)